In [1]:
# 04_decomposition.ipynb
# =============================================================================
# RQ2 -- signal decomposition: intra-risk vs contagion. Node features are held
# fixed; only message passing changes.
#   INTRA-ONLY            : feed-forward MLP on the 23-d company features, no graph.
#   CONTAGION (0-11)      : hetero-GNN propagating over the dense economic ties.
#   CONTAGION (0-19)      : same GNN over all relations (adds sparse financial links).
#
# Beyond the point estimates we add three rigor layers that the prior version
# lacked:
#   (1) EDGE-SHUFFLE PERMUTATION NULL: re-wire the economic edges at random
#       (preserving per-relation edge counts) and re-train. If the contagion
#       uplift is real structure rather than a capacity/attribution artifact,
#       performance must collapse toward the intra-only level under the null.
#   (2) DeLong test for the AUC difference on the SAME test set (intra vs 0-11),
#       which, unlike a paired t-test over seed means, accounts for the
#       correlation of predictions on shared test cases.
#   (3) Bootstrap 95% CIs for the test AUC of each configuration.
# 10 seeds, mask A (strict). Figures grayscale, dpi 600, png + pdf, no caption.
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, norm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from torch_geometric.nn import HeteroConv, SAGEConv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
DEVICE = torch.device("cpu")
SEEDS = list(range(10))
N_PERM = 10


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


def ks_stat(y, s):
    fpr, tpr, _ = roc_curve(y, s); return float(np.max(tpr - fpr))


def metrics(y, s, thr=0.5):
    return {"AUC": roc_auc_score(y, s), "F1": f1_score(y, (s >= thr).astype(int),
            zero_division=0), "KS": ks_stat(y, s)}


# --- DeLong test (fast) ------------------------------------------------------
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(N); out[J] = T
    return out


def delong_auc_var(y_true, s1, s2):
    # Fast DeLong (Sun & Xu 2014), paired on the same test samples.
    # Positives are placed first so the midrank columns align correctly.
    pos = y_true == 1
    sx = np.vstack([s1[pos], s2[pos]])        # (2, m) positive scores
    sy = np.vstack([s1[~pos], s2[~pos]])      # (2, n) negative scores
    m, n = sx.shape[1], sy.shape[1]
    tx = np.array([_midrank(sx[k]) for k in range(2)])
    ty = np.array([_midrank(sy[k]) for k in range(2)])
    txy = np.array([_midrank(np.concatenate([sx[k], sy[k]])) for k in range(2)])
    aucs = txy[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (txy[:, :m] - tx) / n
    v10 = 1.0 - (txy[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    z = (aucs[0] - aucs[1]) / np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1] + 1e-12)
    p = 2 * (1 - norm.cdf(abs(z)))
    return float(aucs[0]), float(aucs[1]), float(p)


def bootstrap_auc_ci(y, s, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    vals = []
    for _ in range(n_boot):
        bi = np.concatenate([rng.choice(pos, len(pos), True), rng.choice(neg, len(neg), True)])
        vals.append(roc_auc_score(y[bi], s[bi]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


# --- Load + preprocess -------------------------------------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
Xc = comp.x.numpy().astype(np.float64)
Xc[:, :2] = np.log1p(np.clip(Xc[:, :2], 0, None))
tr_mask = comp.train_mask.numpy()
Xc = StandardScaler().fit(Xc[tr_mask]).transform(Xc).astype(np.float32)
comp.x = torch.tensor(Xc, dtype=torch.float32)
y = comp.y
trm, vam, tem = comp.train_mask, comp.val_mask, comp.test_mask
EDGE_TYPES = list(data.edge_types)
ET_0_11 = [et for et in EDGE_TYPES if int(et[1].split("_")[1]) <= 11]
ET_0_19 = EDGE_TYPES
IN_DIMS = {"company": comp.x.size(1), "person": data["person"].x.size(1)}


class HeteroGNN(nn.Module):
    def __init__(self, edge_types, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.ets = edge_types; self.drop = dropout
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in IN_DIMS.items()})
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((hidden, hidden), hidden) for et in edge_types},
                       aggr="sum") for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, x_dict, ei_dict):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in x_dict.items()}
        for conv in self.convs:
            h = conv(h, ei_dict)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training)
                 for nt, v in h.items()}
        return self.head(h["company"])


class IntraMLP(nn.Module):
    def __init__(self, d, hidden=64, out=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, out))

    def forward(self, x):
        return self.net(x)


def class_w(mask):
    cls = torch.bincount(y[mask], minlength=2).float()
    return cls.sum() / (2 * cls.clamp(min=1))


def train_eval(kind, edge_dict=None, seed=0, epochs=200, lr=5e-3, wd=5e-4, patience=30,
               return_prob=False):
    torch.manual_seed(seed); np.random.seed(seed)
    w = class_w(trm)
    if kind == "intra":
        model = IntraMLP(comp.x.size(1))
        forward = lambda: model(comp.x)
    else:
        model = HeteroGNN(list(edge_dict.keys()))
        xd = {nt: data[nt].x for nt in data.node_types}
        forward = lambda: model(xd, edge_dict)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_state, wait = -1, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(forward()[trm], y[trm], weight=w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            prob = F.softmax(forward(), 1)[:, 1].numpy()
        va = roc_auc_score(y[vam].numpy(), prob[vam.numpy()])
        if va > best_val:
            best_val, best_state, wait = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        prob = F.softmax(forward(), 1)[:, 1].numpy()
    met = metrics(y[tem.numpy()].numpy(), prob[tem.numpy()])
    if return_prob:
        return met, prob[tem.numpy()]
    return met


# --- Edge dicts --------------------------------------------------------------
ei_0_11 = {et: data[et].edge_index for et in ET_0_11}
ei_0_19 = {et: data[et].edge_index for et in ET_0_19}

# --- 1. Three configurations x 10 seeds --------------------------------------
rows = []
prob_store = {}
for seed in SEEDS:
    m_i, p_i = train_eval("intra", seed=seed, return_prob=True)
    m_c, p_c = train_eval("contagion", ei_0_11, seed=seed, return_prob=True)
    m_a, p_a = train_eval("contagion", ei_0_19, seed=seed, return_prob=True)
    rows += [{"config": "intra", **m_i, "seed": seed},
             {"config": "contagion_0_11", **m_c, "seed": seed},
             {"config": "contagion_0_19", **m_a, "seed": seed}]
    if seed == 0:
        prob_store = {"intra": p_i, "contagion_0_11": p_c, "contagion_0_19": p_a}
raw = pd.DataFrame(rows)
raw.to_csv(TAB / "04_decomposition_raw.csv", index=False)

summary = raw.groupby("config").agg(
    AUC_mean=("AUC", "mean"), AUC_std=("AUC", "std"),
    F1_mean=("F1", "mean"), KS_mean=("KS", "mean")).reset_index().round(4)
order = {"intra": 0, "contagion_0_11": 1, "contagion_0_19": 2}
summary = summary.sort_values("config", key=lambda c: c.map(order)).reset_index(drop=True)
summary.to_csv(TAB / "04_decomposition.csv", index=False)
print("=== RQ2 decomposition (mask A, 10 seeds) ===")
print(summary.to_string(index=False))

intra_auc = summary.set_index("config").loc["intra", "AUC_mean"]
c11_auc = summary.set_index("config").loc["contagion_0_11", "AUC_mean"]
c19_auc = summary.set_index("config").loc["contagion_0_19", "AUC_mean"]
print(f"\ncontagion uplift (0-11 - intra) = {c11_auc - intra_auc:+.3f}")
print(f"sparse financial links (0-19 - 0-11) = {c19_auc - c11_auc:+.3f}")

# paired t-test across seeds
a = raw.pivot(index="seed", columns="config", values="AUC")
t1, p1 = ttest_rel(a["contagion_0_11"], a["intra"])
t2, p2 = ttest_rel(a["contagion_0_19"], a["contagion_0_11"])
print(f"paired t (0-11 vs intra): t={t1:.2f} p={p1:.2e}")
print(f"paired t (0-19 vs 0-11) : t={t2:.2f} p={p2:.3f}")

# --- 2. DeLong + bootstrap on seed 0 test set --------------------------------
yt = y[tem.numpy()].numpy()
auc_i, auc_c, p_delong = delong_auc_var(yt, prob_store["intra"], prob_store["contagion_0_11"])
ci_i = bootstrap_auc_ci(yt, prob_store["intra"])
ci_c = bootstrap_auc_ci(yt, prob_store["contagion_0_11"])
print(f"\nDeLong (intra {auc_i:.3f} vs contagion-0-11 {auc_c:.3f}) p={p_delong:.2e}")
print(f"bootstrap 95% CI  intra        : [{ci_i[0]:.3f}, {ci_i[1]:.3f}]")
print(f"bootstrap 95% CI  contagion0-11: [{ci_c[0]:.3f}, {ci_c[1]:.3f}]")
pd.DataFrame([{"intra_AUC": auc_i, "contagion_AUC": auc_c, "delong_p": p_delong,
               "intra_CI_lo": ci_i[0], "intra_CI_hi": ci_i[1],
               "contagion_CI_lo": ci_c[0], "contagion_CI_hi": ci_c[1]}]).round(5).to_csv(
    TAB / "04_delong_bootstrap.csv", index=False)

# --- 3. Edge-shuffle permutation null (economic ties 0-11) -------------------
def shuffled_edges(seed):
    g = torch.Generator().manual_seed(1000 + seed)
    ed = {}
    for et in ET_0_11:
        ei = data[et].edge_index
        st, _, dt = et
        n_dst = data[dt].num_nodes
        perm_dst = torch.randint(0, n_dst, (ei.size(1),), generator=g)
        ed[et] = torch.stack([ei[0], perm_dst])
    return ed


null_rows = []
for seed in range(N_PERM):
    ed = shuffled_edges(seed)
    m_n = train_eval("contagion", ed, seed=seed)
    null_rows.append({"config": "null_shuffle_0_11", **m_n, "seed": seed})
null_df = pd.DataFrame(null_rows)
null_df.to_csv(TAB / "04_permutation_null.csv", index=False)
null_auc = null_df["AUC"].mean()
print(f"\npermutation null (shuffled 0-11 edges, {N_PERM} draws): "
      f"AUC {null_auc:.3f}+/-{null_df['AUC'].std():.3f}")
print(f"  real contagion {c11_auc:.3f} vs null {null_auc:.3f} "
      f"(drop {c11_auc - null_auc:+.3f}); intra {intra_auc:.3f}")
# empirical p: fraction of null >= real mean
emp_p = float((null_df['AUC'] >= c11_auc).mean())
print(f"  empirical p(null AUC >= real contagion) = {emp_p:.3f}")

# --- 4. Figure: decomposition with null band ---------------------------------
fig, ax = plt.subplots(figsize=(5.6, 3.6))
cfgs = ["intra", "contagion_0_11", "contagion_0_19"]
labels = ["Intra-only\n(no graph)", "Contagion\n(0-11)", "Contagion\n(0-19)"]
means = [summary.set_index("config").loc[c, "AUC_mean"] for c in cfgs]
stds = [summary.set_index("config").loc[c, "AUC_std"] for c in cfgs]
shades = ["#999999", "#333333", "#555555"]
ax.bar(np.arange(3), means, yerr=stds, color=shades, edgecolor="black", linewidth=0.5,
       error_kw={"elinewidth": 0.9, "capsize": 3}, width=0.6)
# null band
ax.axhspan(null_auc - null_df['AUC'].std(), null_auc + null_df['AUC'].std(),
           color="#cccccc", alpha=0.5, zorder=0)
ax.axhline(null_auc, color="black", linestyle=":", linewidth=1.0)
ax.text(2.35, null_auc, "edge-shuffle null", fontsize=7, va="bottom", ha="right")
ax.set_xticks(np.arange(3)); ax.set_xticklabels(labels)
ax.set_ylabel("Test AUC (mask A)")
ax.set_ylim(0.6, 0.8)
savefig(fig, "fig_decomposition")

print("\n04_decomposition complete: RQ2 point estimates + permutation null + DeLong + "
      "bootstrap CIs; 1 figure, 4 tables.")


=== RQ2 decomposition (mask A, 10 seeds) ===
        config  AUC_mean  AUC_std  F1_mean  KS_mean
         intra    0.6862   0.0083   0.6015   0.3119
contagion_0_11    0.7671   0.0136   0.6883   0.4357
contagion_0_19    0.7658   0.0135   0.6723   0.4339

contagion uplift (0-11 - intra) = +0.081
sparse financial links (0-19 - 0-11) = -0.001
paired t (0-11 vs intra): t=14.56 p=1.46e-07
paired t (0-19 vs 0-11) : t=-0.31 p=0.766



DeLong (intra 0.678 vs contagion-0-11 0.775) p=1.68e-06
bootstrap 95% CI  intra        : [0.627, 0.726]
bootstrap 95% CI  contagion0-11: [0.729, 0.815]



permutation null (shuffled 0-11 edges, 10 draws): AUC 0.671+/-0.014
  real contagion 0.767 vs null 0.671 (drop +0.096); intra 0.686
  empirical p(null AUC >= real contagion) = 0.000



04_decomposition complete: RQ2 point estimates + permutation null + DeLong + bootstrap CIs; 1 figure, 4 tables.
